# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [3]:
# 📦 Imports & Logging Setup (Bonus: structured logging)

import re
import json
import logging
import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("SmartAgent")

print("✅ Imports and logging ready")

✅ Imports and logging ready


In [4]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    try:
        expression = expression.strip()
        # Whitelist: only digits, operators, spaces, parentheses, dots
        if not re.match(r'^[\d\s\+\-\*\/\%\(\)\.\^\*\*]+$', expression):
            return "Error: Invalid characters in expression"
        result = eval(expression)
        logger.info(f"Calculator: '{expression}' = {result}")
        return str(result)
    except ZeroDivisionError:
        logger.warning(f"Calculator: Division by zero in '{expression}'")
        return "Error: Division by zero"
    except Exception as e:
        logger.error(f"Calculator error: {e}")
        return "Error in calculation"


In [5]:
# 🛠️ TOOL 2: Keyword Extractor

STOP_WORDS = {
    "from", "this", "that", "with", "have", "will", "been",
    "they", "them", "their", "there", "these", "those", "about",
    "what", "which", "when", "where", "also", "just", "some"
}

def extract_keywords(text: str) -> list:
    """Extract meaningful keywords from text, filtering stop words."""
    try:
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        # Preserve insertion order, deduplicate, filter stop words & short words
        keywords = list(dict.fromkeys(
            w.lower() for w in words
            if len(w) > 4 and w.lower() not in STOP_WORDS
        ))
        logger.info(f"Keywords extracted: {keywords[:5]}")
        return keywords[:5]
    except Exception as e:
        logger.error(f"Keyword extraction error: {e}")
        return []


In [6]:
# 🛠️ TOOL 3 (Bonus): Text Summarizer

def summarize(text: str) -> str:
    """Summarize text by returning the first two meaningful sentences."""
    try:
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        sentences = [s for s in sentences if len(s) > 20]
        if not sentences:
            return "No meaningful content to summarize."
        summary = sentences[0]
        if len(sentences) > 1:
            summary += " " + sentences[1]
        logger.info(f"Summarizer produced {len(sentences)}-sentence summary")
        return summary
    except Exception as e:
        logger.error(f"Summarizer error: {e}")
        return "Error in summarization"


# 🛠️ TOOL 4 (Bonus): Unit Converter

UNIT_CONVERSIONS = {
    ("km",        "miles"):      lambda x: x * 0.621371,
    ("miles",     "km"):         lambda x: x * 1.60934,
    ("kg",        "lbs"):        lambda x: x * 2.20462,
    ("lbs",       "kg"):         lambda x: x * 0.453592,
    ("celsius",   "fahrenheit"): lambda x: x * 9/5 + 32,
    ("fahrenheit","celsius"):    lambda x: (x - 32) * 5/9,
}

def unit_converter(query: str) -> str:
    """Convert between common units. E.g. 'convert 100 km to miles'."""
    try:
        match = re.search(r'([\d\.]+)\s*(\w+)\s+(?:to|in)\s+(\w+)', query, re.IGNORECASE)
        if not match:
            return "Could not parse conversion. Try: 'convert 10 km to miles'"
        value    = float(match.group(1))
        from_unit = match.group(2).lower()
        to_unit   = match.group(3).lower()
        key = (from_unit, to_unit)
        if key not in UNIT_CONVERSIONS:
            return f"Conversion from {from_unit} to {to_unit} is not supported."
        result = UNIT_CONVERSIONS[key](value)
        logger.info(f"Converter: {value} {from_unit} → {result:.4f} {to_unit}")
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    except Exception as e:
        logger.error(f"Unit converter error: {e}")
        return "Error in unit conversion"

print("✅ All tools defined")

✅ All tools defined


## 🤖 Agent Logic

**Routing strategy (improved — Bonus):**

| Pattern matched in query | Tool used |
|---|---|
| `convert / conversion` | Unit Converter |
| `calculate / compute / solve` + digits | Calculator |
| `what is` + number/expression | Calculator |
| `keywords / extract / key terms` | Keyword Extractor |
| `summarize / summary / brief` | Summarizer |
| Everything else | General response |

Unit Converter is checked **before** Calculator to avoid overlap (e.g. "convert 10 km to miles" contains digits).


In [7]:
# 🤖 AGENT FUNCTION

# --- Routing patterns (Bonus: regex-based, more robust than simple 'in') ---
CALC_TRIGGER    = re.compile(r'\b(calculate|compute|solve|eval|how much is)\b', re.I)
WHAT_IS_CALC    = re.compile(r'\bwhat is\b\s+[\d\(]', re.I)   # 'what is 2+2' → calc
KEYWORD_TRIGGER = re.compile(r'\b(keywords?|key terms?|extract|important words?)\b', re.I)
SUMM_TRIGGER    = re.compile(r'\b(summarize|summary|tldr|brief|shorten)\b', re.I)
CONV_TRIGGER    = re.compile(r'\b(convert|conversion)\b', re.I)


def _extract_math_expression(query: str) -> str:
    """Strip intent words and pull out the numeric expression."""
    cleaned = CALC_TRIGGER.sub('', query)
    cleaned = re.sub(r'\bwhat is\b', '', cleaned, flags=re.I).strip()
    # Match expressions starting with digit or '('
    match = re.search(r'[\d\(][\d\s\+\-\*\/\%\.\(\)\*\^]*[\d\)]', cleaned)
    return match.group(0).strip() if match else cleaned.strip()


def _extract_keyword_text(query: str) -> str:
    """Strip intent phrase to isolate the text for keyword extraction."""
    cleaned = KEYWORD_TRIGGER.sub('', query, count=1)
    cleaned = re.sub(r'^\s*(from|in|of|for)\s+', '', cleaned, flags=re.I)
    return cleaned.strip()


def _extract_summary_text(query: str) -> str:
    """Strip summarize intent phrase to isolate the target text."""
    cleaned = SUMM_TRIGGER.sub('', query, count=1)
    cleaned = re.sub(r'^[:\s]+', '', cleaned).strip()
    return cleaned


def agent(query: str) -> dict:
    """
    Single-Agent Smart Assistant.

    Routes the query to the appropriate tool and returns structured output:
        {"type": "calculation|keywords|summary|conversion|general|error",
         "result": <value>}
    """
    # --- Input validation ---
    if not isinstance(query, str) or not query.strip():
        logger.warning("Empty or invalid query received.")
        return {"type": "error", "result": "Query must be a non-empty string."}

    query = query.strip()
    logger.info(f"Agent received: '{query}'")

    try:
        # 1. Unit Converter — checked first (contains digits that could fool calc)
        if CONV_TRIGGER.search(query):
            logger.info("Route → Unit Converter")
            return {"type": "conversion", "result": unit_converter(query)}

        # 2. Calculator
        elif CALC_TRIGGER.search(query) or WHAT_IS_CALC.search(query):
            expr = _extract_math_expression(query)
            logger.info(f"Route → Calculator | expr: '{expr}'")
            return {"type": "calculation", "result": calculator(expr)}

        # 3. Keyword Extractor
        elif KEYWORD_TRIGGER.search(query):
            text = _extract_keyword_text(query)
            logger.info(f"Route → Keyword Extractor | text: '{text}'")
            return {"type": "keywords", "result": extract_keywords(text)}

        # 4. Summarizer (Bonus)
        elif SUMM_TRIGGER.search(query):
            text = _extract_summary_text(query)
            logger.info(f"Route → Summarizer | text: '{text[:40]}...'")
            return {"type": "summary", "result": summarize(text)}

        # 5. General fallback
        else:
            logger.info("Route → General Response")
            return {
                "type": "general",
                "result": (
                    f"I received your query: '{query}'. "
                    "I can help with: calculations, keyword extraction, "
                    "summarization, and unit conversions."
                )
            }

    except Exception as e:
        logger.error(f"Unexpected agent error: {e}")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

print("✅ Agent function ready")

✅ Agent function ready


## 📦 Expected Output Format

```json
{
  "type": "calculation | keywords | summary | conversion | general | error",
  "result": "..."
}
```

In [8]:
# 🧪 Test Cases

queries = [
    # --- Required (original 3) ---
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",

    # --- Additional calculator tests ---
    "compute (100 * 3) / 4",
    "What is 2 ** 10",

    # --- Keyword variations ---
    "Find key terms in deep learning enables computers to learn from data",

    # --- Bonus: Summarizer ---
    "Summarize: Machine learning is a subset of AI. It allows systems to learn from data and improve over time without being explicitly programmed.",

    # --- Bonus: Unit Converter ---
    "Convert 100 km to miles",
    "Convert 37 celsius to fahrenheit",

    # --- Error handling ---
    "Calculate 10 / 0",
    "",
]

print("=" * 60)
for q in queries:
    print(f"Query   : {q!r}")
    response = agent(q)
    print(f"Response: {json.dumps(response, indent=2)}")
    print("-" * 60)

2026-06-28 13:25:30,635 [INFO] Agent received: 'Calculate 20 + 5'
2026-06-28 13:25:30,637 [INFO] Route → Calculator | expr: '20 + 5'
2026-06-28 13:25:30,638 [INFO] Calculator: '20 + 5' = 25
2026-06-28 13:25:30,641 [INFO] Agent received: 'Extract keywords from Artificial Intelligence is transforming industries'
2026-06-28 13:25:30,641 [INFO] Route → Keyword Extractor | text: 'keywords from Artificial Intelligence is transforming industries'
2026-06-28 13:25:30,646 [INFO] Keywords extracted: ['keywords', 'artificial', 'intelligence', 'transforming', 'industries']
2026-06-28 13:25:30,649 [INFO] Agent received: 'What is machine learning?'
2026-06-28 13:25:30,652 [INFO] Route → General Response
2026-06-28 13:25:30,655 [INFO] Agent received: 'compute (100 * 3) / 4'
2026-06-28 13:25:30,657 [INFO] Route → Calculator | expr: '(100 * 3) / 4'
2026-06-28 13:25:30,659 [INFO] Calculator: '(100 * 3) / 4' = 75.0
2026-06-28 13:25:30,662 [INFO] Agent received: 'What is 2 ** 10'
2026-06-28 13:25:30,666 [

Query   : 'Calculate 20 + 5'
Response: {
  "type": "calculation",
  "result": "25"
}
------------------------------------------------------------
Query   : 'Extract keywords from Artificial Intelligence is transforming industries'
Response: {
  "type": "keywords",
  "result": [
    "keywords",
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ]
}
------------------------------------------------------------
Query   : 'What is machine learning?'
Response: {
  "type": "general",
  "result": "I received your query: 'What is machine learning?'. I can help with: calculations, keyword extraction, summarization, and unit conversions."
}
------------------------------------------------------------
Query   : 'compute (100 * 3) / 4'
Response: {
  "type": "calculation",
  "result": "75.0"
}
------------------------------------------------------------
Query   : 'What is 2 ** 10'
Response: {
  "type": "calculation",
  "result": "1024"
}
------------------------------------

2026-06-28 13:25:30,699 [INFO] Converter: 37.0 celsius → 98.6000 fahrenheit
2026-06-28 13:25:30,704 [INFO] Agent received: 'Calculate 10 / 0'
2026-06-28 13:25:30,706 [INFO] Route → Calculator | expr: '10 / 0'
2026-06-28 13:25:30,707 [WARNING] Calculator: Division by zero in '10 / 0'
2026-06-28 13:25:30,708 [WARNING] Empty or invalid query received.


Response: {
  "type": "conversion",
  "result": "37.0 celsius = 98.6000 fahrenheit"
}
------------------------------------------------------------
Query   : 'Calculate 10 / 0'
Response: {
  "type": "calculation",
  "result": "Error: Division by zero"
}
------------------------------------------------------------
Query   : ''
Response: {
  "type": "error",
  "result": "Query must be a non-empty string."
}
------------------------------------------------------------


In [ ]:
# 🎯 Interactive Mode (VS Code compatible — no ipywidgets needed)

# Suppress the deprecation warning
import warnings
warnings.filterwarnings("ignore")

print("🤖 Smart Agent — Interactive Mode")
print("=" * 50)
print("Commands: 'calculate 5 * 8', 'extract keywords from ...', 'convert 10 km to miles'")
print("Type 'exit' to stop.\n")

chat_history = []

def run_agent_session():
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Session ended.")
            break

        if user_input.lower() == "exit":
            print("Goodbye! 👋")
            break

        if not user_input:
            print("⚠️  Please enter a query.\n")
            continue

        response = agent(user_input)
        chat_history.append({"query": user_input, "response": response})

        print(f"🤖 Agent: {json.dumps(response, indent=2)}\n")
        print("-" * 50)

run_agent_session()

# Show full session history after exit
if chat_history:
    print("\n📋 Session Summary:")
    for i, turn in enumerate(chat_history, 1):
        print(f"  {i}. [{turn['response']['type']}] {turn['query']!r} → {turn['response']['result']}")

🤖 Smart Agent — Interactive Mode
Commands: 'calculate 5 * 8', 'extract keywords from ...', 'convert 10 km to miles'
Type 'exit' to stop.

⚠️  Please enter a query.



2026-06-28 13:26:12,363 [INFO] Agent received: '9/0'
2026-06-28 13:26:12,365 [INFO] Route → General Response


🤖 Agent: {
  "type": "general",
  "result": "I received your query: '9/0'. I can help with: calculations, keyword extraction, summarization, and unit conversions."
}

--------------------------------------------------


2026-06-28 13:26:26,076 [INFO] Agent received: 'calculate 9/0'
2026-06-28 13:26:26,077 [INFO] Route → Calculator | expr: '9/0'
2026-06-28 13:26:26,079 [WARNING] Calculator: Division by zero in '9/0'


🤖 Agent: {
  "type": "calculation",
  "result": "Error: Division by zero"
}

--------------------------------------------------


2026-06-28 13:26:41,633 [INFO] Agent received: 'Calculate 9 / 0'
2026-06-28 13:26:41,634 [INFO] Route → Calculator | expr: '9 / 0'
2026-06-28 13:26:41,635 [WARNING] Calculator: Division by zero in '9 / 0'


🤖 Agent: {
  "type": "calculation",
  "result": "Error: Division by zero"
}

--------------------------------------------------


2026-06-28 13:26:53,370 [INFO] Agent received: 'Summarize: Deep learning is a branch of machine learning. It uses neural networks with many layers to model complex patterns in data.'
2026-06-28 13:26:53,372 [INFO] Route → Summarizer | text: 'Deep learning is a branch of machine lea...'
2026-06-28 13:26:53,373 [INFO] Summarizer produced 2-sentence summary


🤖 Agent: {
  "type": "summary",
  "result": "Deep learning is a branch of machine learning. It uses neural networks with many layers to model complex patterns in data."
}

--------------------------------------------------
